In [80]:

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [81]:
from motor_ingesta.motor_ingesta import MotorIngesta
from pyspark.sql import DataFrame as DF, functions as F

In [43]:
json_path="abfss://datos@masterap001sta.dfs.core.windows.net/2023-01-01.json"
flights_day_df = spark.read.option("multiLine", "true").json(json_path)

In [44]:
flights_day_df.cache()

DataFrame[FlightDate: string, flights_list: array<struct<ActualElapsedTime:double,AirTime:double,CRSArrTime:bigint,CRSDepTime:bigint,CRSElapsedTime:double,CancellationCode:string,Cancelled:double,Distance:double,DistanceGroup:bigint,Diverted:double,FirstDepTime:bigint,Flights:double,LongestAddGTime:double,Tail_Number:string,TaxiIn:double,TaxiOut:double,TotalAddGTime:double,WheelsOff:bigint,WheelsOn:bigint,airline_info:struct<DOT_ID_Reporting_Airline:bigint,Flight_Number_Reporting_Airline:bigint,IATA_CODE_Reporting_Airline:string,Reporting_Airline:string>,arrival_performance:struct<ArrDel15:double,ArrDelay:double,ArrDelayMinutes:double,ArrTime:bigint,ArrTimeBlk:string,ArrivalDelayGroups:bigint>,delay_info:struct<CarrierDelay:double,LateAircraftDelay:double,NASDelay:double,SecurityDelay:double,WeatherDelay:double>,departure_performance:struct<DepDel15:double,DepDelay:double,DepDelayMinutes:double,DepTime:bigint,DepTimeBlk:string,DepartureDelayGroups:bigint>,destination_info:struct<Dest:s

In [45]:
def aplana_df(df: DF) -> DF:
    """
    Aplana un DataFrame de Spark que tenga columnas de tipo array y de tipo estructura.

    :param df: DataFrame de Spark que contiene columnas de tipo array o columnas de tipo estructura, incluyendo
               cualquier nivel de anidamiento y también arrays de estructuras. Asumimos que los nombres de los
               campos anidados son todos distintos entre sí, y no van a coincidir cuando sean aplanados.
    :return: DataFrame de Spark donde todas las columnas de tipo array han sido explotadas y las estructuras
             han sido aplanadas recursivamente.
    """
    to_select = []
    schema = df.schema.jsonValue()
    fields = schema["fields"]
    recurse = False

    for f in fields:
        if f["type"].__class__.__name__ != "dict":
            to_select.append(f["name"])
        else:
            if f["type"]["type"] == "array":
                to_select.append(F.explode(f["name"]).alias(f["name"]))
                recurse = True
            elif f["type"]["type"] == "struct":
                to_select.append(f"{f['name']}.*")
                recurse = True

    new_df = df.select(*to_select)
    return MotorIngesta.aplana_df(new_df) if recurse else new_df

In [46]:
aplanado_df = aplana_df(flights_day_df)


In [47]:
aplanado_df

DataFrame[FlightDate: string, ActualElapsedTime: double, AirTime: double, CRSArrTime: bigint, CRSDepTime: bigint, CRSElapsedTime: double, CancellationCode: string, Cancelled: double, Distance: double, DistanceGroup: bigint, Diverted: double, FirstDepTime: bigint, Flights: double, LongestAddGTime: double, Tail_Number: string, TaxiIn: double, TaxiOut: double, TotalAddGTime: double, WheelsOff: bigint, WheelsOn: bigint, DOT_ID_Reporting_Airline: bigint, Flight_Number_Reporting_Airline: bigint, IATA_CODE_Reporting_Airline: string, Reporting_Airline: string, ArrDel15: double, ArrDelay: double, ArrDelayMinutes: double, ArrTime: bigint, ArrTimeBlk: string, ArrivalDelayGroups: bigint, CarrierDelay: double, LateAircraftDelay: double, NASDelay: double, SecurityDelay: double, WeatherDelay: double, DepDel15: double, DepDelay: double, DepDelayMinutes: double, DepTime: bigint, DepTimeBlk: string, DepartureDelayGroups: bigint, Dest: string, DestAirportID: bigint, DestAirportSeqID: bigint, DestCityMark

In [49]:
import json
from pathlib import Path


root_path = Path.cwd().parent
config_file= str(root_path/"config"/"config.json")


In [50]:
print(config_file)

C:\repos\repos_master\repo_spark\spark-tarea-final\config\config.json


In [51]:
with open(config_file, "r") as f:
    config = json.load(f)

In [52]:
lista_obj_column = [F.col(diccionario["name"]).cast(diccionario["type"]).alias(diccionario["name"], metadata={"comment": diccionario["comment"]})
                                  for diccionario in config["data_columns"] ]
resultado_df = aplanado_df.select(*lista_obj_column)

In [54]:
resultado_df.head(5)

[Row(FlightDate=datetime.date(2023, 1, 1), Reporting_Airline='9E', OriginAirportID=12884, Origin='LAN', OriginCityName='Lansing, MI', OriginState='MI', DestAirportID=11433, Dest='DTW', DestCityName='Detroit, MI', DestState='MI', DepDelay=-6, DepTime=1136, ArrDelay=-22, ArrTime=1215, Cancelled=False, Diverted=False, AirTime=21, Distance=74),
 Row(FlightDate=datetime.date(2023, 1, 1), Reporting_Airline='DL', OriginAirportID=13487, Origin='MSP', OriginCityName='Minneapolis, MN', OriginState='MN', DestAirportID=13495, Dest='MSY', DestCityName='New Orleans, LA', DestState='LA', DepDelay=-1, DepTime=1015, ArrDelay=-17, ArrTime=1244, Cancelled=False, Diverted=False, AirTime=133, Distance=1039),
 Row(FlightDate=datetime.date(2023, 1, 1), Reporting_Airline='AS', OriginAirportID=14747, Origin='SEA', OriginCityName='Seattle, WA', OriginState='WA', DestAirportID=14679, Dest='SAN', DestCityName='San Diego, CA', DestState='CA', DepDelay=1, DepTime=1946, ArrDelay=8, ArrTime=2241, Cancelled=False, Div

In [82]:
from motor_ingesta.agregaciones import aniade_hora_utc, aniade_intervalos_por_aeropuerto
import pandas as pd

In [83]:
path_timezones =  Path.cwd() / "resources" / "timezones.csv"
timezones_pd = pd.read_csv(path_timezones)
timezones_df = spark.createDataFrame(timezones_pd)

In [85]:
timezones_df.head(5)

[Row(iata_code='AAA', iana_tz='Pacific/Tahiti', windows_tz='Hawaiian Standard Time'), Row(iata_code='AAB', iana_tz='Australia/Brisbane', windows_tz='E. Australia Standard Time'), Row(iata_code='AAC', iana_tz='Africa/Cairo', windows_tz='Egypt Standard Time'), Row(iata_code='AAD', iana_tz='Africa/Mogadishu', windows_tz='E. Africa Standard Time'), Row(iata_code='AAE', iana_tz='Africa/Algiers', windows_tz='W. Central Africa Standard Time')]
